# 4. Seguridad – Grants, usuarios y grupos
Notebook de demostración del modelo de permisos de Unity Catalog del proyecto.

| Principal | Rol | Permisos |
|---|---|---|
| `data_engineers` (data_engineer1, data_engineer2) | Mantienen el ETL | Catálogo; bronze, silver y golden: lectura, escritura y creación de tablas; external locations |
| `analistas` (analista1, analista2) | Analizan datos limpios | silver y golden: solo lectura |
| `consumidores_bi` (consumidor_bi1, consumidor_bi2) | Power BI / dashboards | Solo las tablas golden |
| `analista1@outlook.com` (individual) | Excepción puntual | Lectura del catálogo de sucursales en bronze |

> Los grupos y usuarios se crean en la **Account Console** antes de ejecutar este notebook.

## 1. Usuarios y grupos

In [0]:
SHOW USERS;

In [0]:
SHOW GROUPS;

Creación de grupos y asignación de usuarios desde SQL (alternativa a la Account Console).
Se dejan comentados porque los grupos del proyecto se administran a nivel de cuenta.

In [0]:
-- CREATE GROUP `data_engineers`;
-- CREATE GROUP `analistas`;
-- CREATE GROUP `consumidores_bi`;

-- ALTER GROUP `data_engineers` ADD USER `data_engineer1@outlook.com`;
-- ALTER GROUP `data_engineers` ADD USER `data_engineer2@outlook.com`;
-- ALTER GROUP `analistas` ADD USER `analista1@outlook.com`;
-- ALTER GROUP `analistas` ADD USER `analista2@outlook.com`;
-- ALTER GROUP `consumidores_bi` ADD USER `consumidor_bi1@outlook.com`;
-- ALTER GROUP `consumidores_bi` ADD USER `consumidor_bi2@outlook.com`;

## 2. Grants

### Catálogo

In [0]:
GRANT USE CATALOG ON CATALOG vehiculos TO `data_engineers`;
GRANT USE CATALOG ON CATALOG vehiculos TO `analistas`;
GRANT USE CATALOG ON CATALOG vehiculos TO `consumidores_bi`;

### Data Engineers – esquemas

In [0]:
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE ON SCHEMA vehiculos.bronze TO `data_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE ON SCHEMA vehiculos.silver TO `data_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE ON SCHEMA vehiculos.golden TO `data_engineers`;

### Data Engineers – external locations

In [0]:
GRANT READ FILES ON EXTERNAL LOCATION `exlt-raw` TO `data_engineers`;
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION `exlt-bronze` TO `data_engineers`;
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION `exlt-silver` TO `data_engineers`;
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION `exlt-golden` TO `data_engineers`;

### Analistas – silver y golden (solo lectura)

In [0]:
GRANT USE SCHEMA, SELECT ON SCHEMA vehiculos.silver TO `analistas`;
GRANT USE SCHEMA, SELECT ON SCHEMA vehiculos.golden TO `analistas`;

### Consumidores BI – solo tablas golden

In [0]:
GRANT USE SCHEMA ON SCHEMA vehiculos.golden TO `consumidores_bi`;

GRANT SELECT ON TABLE vehiculos.golden.ventas_mensuales_estado TO `consumidores_bi`;
GRANT SELECT ON TABLE vehiculos.golden.ranking_modelos TO `consumidores_bi`;
GRANT SELECT ON TABLE vehiculos.golden.kpi_concesionarios TO `consumidores_bi`;
GRANT SELECT ON TABLE vehiculos.golden.segmentos_clientes TO `consumidores_bi`;

### Usuario individual

In [0]:
GRANT USE SCHEMA ON SCHEMA vehiculos.bronze TO `analista1@outlook.com`;
GRANT SELECT ON TABLE vehiculos.bronze.dealers TO `analista1@outlook.com`;

## 3. Auditoría de permisos

In [0]:
SHOW GRANTS ON CATALOG vehiculos;

In [0]:
SHOW GRANTS ON SCHEMA vehiculos.bronze;

In [0]:
SHOW GRANTS ON SCHEMA vehiculos.silver;

In [0]:
SHOW GRANTS ON SCHEMA vehiculos.golden;

In [0]:
SHOW GRANTS ON TABLE vehiculos.golden.kpi_concesionarios;

In [0]:
SHOW GRANTS `analista1@outlook.com` ON TABLE vehiculos.bronze.dealers;

In [0]:
SHOW EXTERNAL LOCATIONS;

In [0]:
SHOW GRANTS ON EXTERNAL LOCATION `exlt-raw`;

In [0]:
SHOW GRANTS ON EXTERNAL LOCATION `exlt-bronze`;

## 4. Revocación de permisos
Ejemplo: se retira el acceso individual de `analista1` a bronze y se valida.

In [0]:
REVOKE SELECT ON TABLE vehiculos.bronze.dealers FROM `analista1@outlook.com`;
REVOKE USE SCHEMA ON SCHEMA vehiculos.bronze FROM `analista1@outlook.com`;

In [0]:
SHOW GRANTS ON SCHEMA vehiculos.bronze;

Se restituye el permiso para dejar el ambiente igual que el workflow.

In [0]:
GRANT USE SCHEMA ON SCHEMA vehiculos.bronze TO `analista1@outlook.com`;
GRANT SELECT ON TABLE vehiculos.bronze.dealers TO `analista1@outlook.com`;

## 5. Ownership (opcional)
Para que el grupo `data_engineers` sea dueño de los esquemas del ETL. Comentado: si se transfiere el ownership, el usuario que ejecuta el workflow debe pertenecer al grupo.

In [0]:
-- ALTER SCHEMA vehiculos.bronze OWNER TO `data_engineers`;
-- ALTER SCHEMA vehiculos.silver OWNER TO `data_engineers`;
-- ALTER SCHEMA vehiculos.golden OWNER TO `data_engineers`;